# Impressão 3D na Engenharia Mecânica (Parte 2) — Predição de Resistência à Tração e Gêmeos Digitais

## Objetivo de Engenharia: Evitar Ensaios Destrutivos com um Gêmeo Digital

Romper dezenas de corpos de prova em uma máquina universal de ensaios de tração consome tempo, filamento e horas de laboratório. Nosso objetivo é construir uma **Rede Neural Artificial de Regressão** capaz de prever a **Resistência à Tração (`tension_strenght`, em MPa)** exclusivamente a partir dos **9 parâmetros de configuração da impressora** (antes da peça ser fabricada!).

Também aprenderemos uma lição crítica da vida real de laboratório: **como treinar Redes Neurais quando o conjunto experimental é pequeno (`N = 50` amostras)** sem cair em armadilhas de não convergência ou *overfitting*!

---

## 1. Carregamento e Matriz de Correlação de Engenharia

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")

# Lê e codifica as duas colunas textuais
df = pd.read_csv("data.csv", sep=",")
df["material"] = df["material"].map({"abs": 0, "pla": 1})
df["infill_pattern"] = df["infill_pattern"].map({"grid": 0, "honeycomb": 1})

# Mapa de calor da correlação linear de Pearson (-1 a +1) entre todas as variáveis
plt.figure(figsize=(10, 7.5))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Matriz de Correlação: Parâmetros de Impressão vs. Propriedades Mecânicas")
plt.tight_layout()
plt.show()

**Leitura Física da Matriz de Correlação:**
Olhe para a linha `tension_strenght` (nosso alvo):
1. **`wall_thickness` (`+0.40`) e `infill_density` (`+0.36`):** Quanto mais espessa a parede externa e mais denso o recheio interno da peça, maior a área resistente efetiva suportando a tração!
2. **`elongation` (`+0.84`):** Alta correlação, porém **não pode entrar em `X`**! Assim como `roughness`, o alongamento até a ruptura só é conhecido *depois* do ensaio destrutivo — incluí-lo nas entradas seria **Vazamento de Dados (*Data Leakage*)**!

---

## 2. Por que `solver='adam'` Sofre em Tabelas Pequenas (`N = 50`) e Como `lbfgs` + Regularização Resolvem?

Se separarmos `80%` para treino (`40` peças) e `20%` para teste (`10` peças):
- O otimizador estocástico padrão `solver='adam'` foi projetado para milhares de amostras; em apenas `40` pontos ele caminha devagar (disparando `ConvergenceWarning`) ou oscila.
- Já o otimizador **`solver='lbfgs'`** (método quase-Newton que aproxima a curvatura de segunda ordem / matriz Hessiana) aliado a uma **regularização `alpha=1.0`** (que freia pesos exagerados contra *overfitting*) converge sem avisos em menos de 1 segundo!

Veja a diferença lado a lado:

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Entradas X = APENAS parâmetros configurados ANTES da impressão!
X = df.drop(["tension_strenght", "roughness", "elongation"], axis=1).to_numpy()
y = df["tension_strenght"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Modelo calibrado para ensaios laboratoriais compactos (lbfgs + regularização L2 alpha=2.0)
mlp_otimizado = MLPRegressor(
    hidden_layer_sizes=(12, 6),
    activation="relu",
    solver="lbfgs",
    alpha=2.0,          # Penalização L2 evita decorar ruído das 40 peças de treino!
    max_iter=2000,
    random_state=42,
)
mlp_otimizado.fit(X_train_sc, y_train)

y_pred = mlp_otimizado.predict(X_test_sc)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Erro Médio Absoluto (MAE) no Teste: {mae:.2f} MPa")
print(f"Coeficiente de Determinação (R²) no Teste: {r2:.4f}")

# Validação Cruzada de 5 Dobras (K-Fold) usando Pipeline (escala cada dobra sem vazamento!)
pipeline_cv = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(12, 6), activation="relu", solver="lbfgs", alpha=2.0, max_iter=2000, random_state=42),
)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
maes_cv = -cross_val_score(pipeline_cv, X, y, cv=kf, scoring="neg_mean_absolute_error")
print(f"MAE Médio em Validação Cruzada 5-Fold (todas as 50 peças): {maes_cv.mean():.2f} ± {maes_cv.std():.2f} MPa")

**Por que usamos Validação Cruzada (`5-Fold`)?** Quando o dataset tem apenas `50` ensaios, testar em apenas `10` peças depende muito da "sorte" de quais 10 peças caíram no teste. Na **Validação Cruzada de 5 Dobras**, dividimos as 50 peças em 5 lotes de 10 e rodamos 5 treinos independentes (sempre escalonando dentro de cada dobra via `make_pipeline`), obtendo uma métrica de erro muito mais confiável!

Vamos plotar o gráfico de paridade **Resistência Real vs. Predita pela RNA**:

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, color="darkblue", s=75, alpha=0.8, label="Corpos de Prova (Teste)")
lim_min, lim_max = min(y.min(), y_pred.min()) - 2, max(y.max(), y_pred.max()) + 2
plt.plot([lim_min, lim_max], [lim_min, lim_max], "r--", linewidth=2, label="Predição Perfeita (y = d)")

plt.title(f"Gêmeo Digital FDM — Resistência à Tração (MAE = {mae:.2f} MPa | R² = {r2:.2f})")
plt.xlabel("Resistência Medida no Ensaio de Tração (MPa)")
plt.ylabel("Resistência Estimada pela Rede Neural (MPa)")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## Aplicações Diretas na Engenharia Industrial
1. **Redução de Ensaios Destrutivos:** Triagem virtual de centenas de combinações de fatiamento (*slicer*) antes de gastar filamento e horas de máquina.
2. **Otimização Multiobjetivo:** Encontrar a menor densidade de preenchimento (`infill_density` = menor peso da peça para indústria aeroespacial/automotiva) que ainda atenda à tensão mínima de projeto com fator de segurança!
3. **Gêmeo Digital (*Digital Twin*):** Modelo virtual embarcado no software fatiador alertando o projetista caso a configuração escolhida gere peça frágil.

---

## 3. Exercícios Práticos de Fixação

### Exercício 1: Simulando Otimização de Peso no Gêmeo Digital
Usando `mlp_otimizado` e `scaler` já treinados, mantenha fixos os parâmetros de uma peça de `PLA` (`layer_height=0.06`, `wall_thickness=5`, `infill_pattern=0`, `nozzle_temperature=210`, `bed_temperature=60`, `print_speed=60`, `material=1`, `fan_speed=100`) e varie `infill_density` em `[20, 40, 60, 80, 100]`%. Qual a menor densidade que entrega pelo menos **`25 MPa`** previstos?

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
# Ordem exata das 9 colunas em X:
# [layer_height, wall_thickness, infill_density, infill_pattern, nozzle_temp, bed_temp, print_speed, material, fan_speed]
for dens in [20, 40, 60, 80, 100]:
    config = np.array([[0.06, 5, dens, 0, 210, 60, 60, 1, 100]], dtype=float)
    tracao_prevista = mlp_otimizado.predict(scaler.transform(config))[0]
    status = "ATENDIDO (>= 25 MPa)" if tracao_prevista >= 25.0 else "Abaixo da meta"
    print(f"Preenchimento = {dens:3d}%  -->  Tração estimada: {tracao_prevista:5.2f} MPa  [{status}]")
```

</details>

### Exercício 2: Predizendo Rugosidade Superficial (`roughness`)
Adapte o código para treinar um novo `MLPRegressor` prevendo agora a **rugosidade superficial (`roughness`, em µm)** a partir dos mesmos 9 parâmetros de impressão e calcule o `MAE` (em µm) e o `R²` sobre o conjunto de teste!

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
y_rug = df["roughness"].to_numpy()
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y_rug, test_size=0.20, random_state=42)

sc_r = StandardScaler()
mlp_rug = MLPRegressor(hidden_layer_sizes=(12, 6), activation="relu", solver="lbfgs", alpha=2.0, max_iter=2000, random_state=42)
mlp_rug.fit(sc_r.fit_transform(X_tr_r), y_tr_r)

pred_rug = mlp_rug.predict(sc_r.transform(X_te_r))
print(f"Predição de Rugosidade -> MAE: {mean_absolute_error(y_te_r, pred_rug):.2f} µm | R²: {r2_score(y_te_r, pred_rug):.4f}")
```

</details>